# Seq Len Hidden State Debug

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.experiments.model_benchmarks.seq_len_debug_utils import (
    auto_select_position_metric_layers,
    plot_avg_metric_per_layer_per_head,
    plot_position_metric_per_layer,
    plot_recurrent_metric_per_head,
    run_position_metric_tracking,
    run_hidden_state_tracking,
    summarize_hidden_state_by_seqlen,
)
from pfns.utils import get_default_device
from pathlib import Path

plt.rcParams['figure.dpi'] = 400


## Config


In [ ]:
EXPERIMENT = {
    'name': 'seq_len_hidden_state_debug',
    'num_classes': 5,
    'num_features': 10,
    'num_test_samples': 100,
    'num_repetitions': 10,
    'data_generation_seed': 42,
    'seqlen_list': [250, 500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000],
}

MODEL_NAMES = [
    'Linear_Attention_Non_Causal',
    'Linear_Attention_Causal_Comb_ST',
    #'Oracle_Hidden_State_Linear_Attention_Non_Causal_base'
    # 'GLA_Comb_ST'
]

TRAINING_CONTEXT_LENGTH = 1_000

# Toggle between the original per-run traces and seq-len violin distributions.
HIDDEN_STATE_PLOT_MODE = 'violin'  # 'violin' or 'individual_runs'
HIDDEN_STATE_DISTRIBUTION_ALPHA = 0.3
HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC = 0.4

HIDDEN_STATE_CACHE_DIR = Path("hidden_state_cache")
HIDDEN_STATE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_STATE_CACHE_OVERWRITE = False

device = str(get_default_device())
models_to_compare = get_models_from_names(MODEL_NAMES)

print(f'Device: {device}')
print(f'Models: {list(models_to_compare)}')
print(f'Repetitions: {EXPERIMENT["num_repetitions"]}')
print(f'Seq lens: {EXPERIMENT["seqlen_list"]}')
print(f'Hidden-state plot mode: {HIDDEN_STATE_PLOT_MODE}')

## Run Hidden-State Tracking


In [ ]:
from pathlib import Path

import pandas as pd

from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import functional_model_config

HIDDEN_STATE_CACHE_DIR = Path("hidden_state_cache")
HIDDEN_STATE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_STATE_CACHE_OVERWRITE = False

hidden_state_cache_key = experiment_payload_hash(
    experiment_payload={
        "experiment": EXPERIMENT,
        "models_to_compare": {
            model_name: functional_model_config(model_config)
            for model_name, model_config in models_to_compare.items()
        },
    }
)
hidden_state_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{EXPERIMENT['name']}_{hidden_state_cache_key}.pkl"
)

if hidden_state_cache_path.exists() and not HIDDEN_STATE_CACHE_OVERWRITE:
    hidden_state_df = pd.read_pickle(hidden_state_cache_path)
    print(f"Loaded hidden_state_df from cache: {hidden_state_cache_path}")
else:
    hidden_state_df = run_hidden_state_tracking(
        experiment=EXPERIMENT,
        models_to_compare=models_to_compare,
        device=device,
    )
    hidden_state_df.to_pickle(hidden_state_cache_path)
    print(f"Saved hidden_state_df to cache: {hidden_state_cache_path}")

print("hidden_state_df rows:", len(hidden_state_df))
hidden_state_df.head()

hidden_state_df_plot = hidden_state_df
print('hidden_state_df_plot rows:', len(hidden_state_df_plot))
hidden_state_df_plot.head()


## Summaries and Plots


In [ ]:
summary_df = summarize_hidden_state_by_seqlen(hidden_state_df)

print('summary_df rows:', len(summary_df))
summary_df.head()

### Metric Guide
For the custom linear-attention models in this notebook, `recurrent_state = kv_state = \sum_t k_t v_t^\top`.
These sequence-length plots therefore show properties of the cached numerator state only, not `k_sum` and not the query-dependent output `(q^\top KV) / (q^\top k_{\mathrm{sum}} + \varepsilon)`.

- `abs_max_mean`: average largest absolute entry in each cached `kv_state` head matrix.
- `frobenius_norm_mean`: per-head Frobenius norm of the cached `kv_state` matrices; this is the raw numerator-state magnitude.
- `spectral_norm_mean`: per-head spectral norm of the cached `kv_state` matrices; this emphasizes the largest singular direction in the numerator state.
- `effective_rank_mean`: effective rank `exp(H(sigma / ||sigma||_1))` of each cached `kv_state` matrix; this shows how spread out the state energy is across singular directions.
- `stable_rank_mean`: stable rank `||KV||_F^2 / ||KV||_2^2` of each cached `kv_state` matrix; this is another concentration-vs-dispersion view of the same numerator state.

If you want denominator-aware views such as `||k_sum||`, `sqrt(||kv_state||_F^2 + ||k_sum||_2^2)`, or `||kv_state||_F / ||k_sum||_2`, use the later linear-attention-specific diagnostics section.


In [ ]:
all_tensors = sorted(summary_df['tensor_name'].astype(str).unique().tolist())
print(f'Recurrent-state tensors available: {len(all_tensors)}')
for name in all_tensors:
    print(' ', name)

In [ ]:
print('Final-state plots above come from run_hidden_state_tracking and show the raw cached states returned by each evaluated runtime model.')
print('Inference clip variants are included here as separate runtime models, but these plots still show the cached states from incontext_fit rather than query-time applied states.')

shared_plot_kwargs = {
    'plot_mode': HIDDEN_STATE_PLOT_MODE,
    'distribution_alpha': HIDDEN_STATE_DISTRIBUTION_ALPHA,
    'distribution_width_frac': HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC,
    #'log_y': True,
}

generic_plot_specs = [
    #('abs_max', plot_recurrent_metric_per_head, 'Final hidden-state abs max vs sequence length'),
    ('frobenius_norm', plot_recurrent_metric_per_head, 'Final hidden-state Frobenius norm vs sequence length'),
    #('spectral_norm', plot_recurrent_metric_per_head, 'Final hidden-state spectral norm vs sequence length'),
    #('effective_rank', plot_recurrent_metric_per_layer, 'Final hidden-state effective rank vs sequence length'),
    #('stable_rank', plot_recurrent_metric_per_layer, 'Final hidden-state stable rank vs sequence length'),
]

linear_attention_plot_specs = [
    ('k_sum_norm', plot_recurrent_metric_per_head, 'Final K-sum norm vs sequence length'),
    ('joint_hidden_state_norm', plot_recurrent_metric_per_head, 'Final joint hidden-state norm vs sequence length'),
    ('kv_over_ksum_ratio', plot_recurrent_metric_per_head, 'Final KV-over-K-sum ratio vs sequence length'),
]

for section_name, plot_specs in [
    ('Generic final-state plots', generic_plot_specs),
    ('Linear-attention-specific final-state plots', linear_attention_plot_specs),
]:
    print(section_name)
    any_plotted = False
    for metric, plot_fn, title_prefix in plot_specs:
        if metric not in hidden_state_df_plot.columns:
            continue
        metric_values = pd.to_numeric(hidden_state_df_plot[metric], errors='coerce')
        if not metric_values.notna().any():
            continue
        plot_fn(
            hidden_state_df_plot,
            metric=metric,
            title_prefix=title_prefix,
            training_context_length=TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        )
        any_plotted = True
    if not any_plotted:
        print(f'- No finite metrics available for: {section_name}')


print('Per-head metric summaries across sequence lengths:')
for metric, title_prefix in [
    ('effective_rank', 'Effective rank per layer'),
    ('stable_rank', 'Stable rank per layer'),
]:
    if metric not in hidden_state_df_plot.columns:
        continue
    metric_values = pd.to_numeric(hidden_state_df_plot[metric], errors='coerce')
    if not metric_values.notna().any():
        continue
    plot_avg_metric_per_layer_per_head(
        hidden_state_df_plot,
        metric=metric,
        title_prefix=title_prefix,
    )


## Q^T k_sum and State Metrics vs Position

Compare the custom non-causal and causal-train-only linear-attention models by logging
$q^\top k_{\mathrm{sum}}$, $\lVert k_{\mathrm{sum}} \rVert$, and $\lVert kv_{\mathrm{state}} \rVert$
as a function of token position for each layer.
These lower plots are the right place to compare inference-only variants, because those variants change the prediction-time applied state rather than the raw cached final state logged above.


In [ ]:
from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import functional_model_config

POSITION_METRIC_EXPERIMENT = {
    'name': 'seq_len_qk_state_position_debug_with_output_norm_svd_alignment_v4',
    'model_names': [
        'Linear_Attention_Causal_Comb_ST',
        'Linear_Attention_Non_Causal',
    ],
    'seqlen': 16_000,
    'num_repetitions': 1,
    'data_generation_seed': EXPERIMENT['data_generation_seed'],
    'num_classes': EXPERIMENT['num_classes'],
    'num_features': EXPERIMENT['num_features'],
    'num_test_samples': EXPERIMENT['num_test_samples'],
    'log_x': True,
}

POSITION_METRIC_LAYER_SELECTION = list(range(15))  # None -> auto-pick first / middle / last
POSITION_METRIC_CACHE_OVERWRITE = False

position_metric_model_configs = get_models_from_names(POSITION_METRIC_EXPERIMENT['model_names'])
position_metric_cache_key = experiment_payload_hash(
    experiment_payload={
        'experiment': POSITION_METRIC_EXPERIMENT,
        'models_to_compare': {
            model_name: functional_model_config(model_config)
            for model_name, model_config in position_metric_model_configs.items()
        },
    }
)
position_metric_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{POSITION_METRIC_EXPERIMENT['name']}_{position_metric_cache_key}.pkl"
)
position_metric_partial_cache_path = position_metric_cache_path.with_name(
    f'{position_metric_cache_path.stem}_partial.pkl'
)

if position_metric_cache_path.exists() and not POSITION_METRIC_CACHE_OVERWRITE:
    position_metric_df = pd.read_pickle(position_metric_cache_path)
    print(f'Loaded position_metric_df from cache: {position_metric_cache_path}')
else:
    position_metric_df = run_position_metric_tracking(
        experiment=POSITION_METRIC_EXPERIMENT,
        device=device,
        training_context_length=TRAINING_CONTEXT_LENGTH,
        partial_cache_path=position_metric_partial_cache_path,
    )
    position_metric_df.to_pickle(position_metric_cache_path)
    if position_metric_partial_cache_path.exists():
        position_metric_partial_cache_path.unlink()
    print(f'Saved position_metric_df to cache: {position_metric_cache_path}')

print('position_metric_df rows:', len(position_metric_df))
position_metric_df.head()

In [ ]:
available_layers = sorted(position_metric_df['layer_idx'].astype(int).unique().tolist())
selected_layers = (
    POSITION_METRIC_LAYER_SELECTION
    if POSITION_METRIC_LAYER_SELECTION is not None
    else auto_select_position_metric_layers(available_layers)
)
print('Selected layers:', selected_layers)

metric_specs = [
    ('q_dot_k_sum', 'q^T k_sum'),
    ('k_sum_norm', '||k_sum||'),
    ('kv_state_norm', '||kv_state||'),
    ('kv_over_ksum_ratio', '||kv_state|| / ||k_sum||'),
    ('output_norm', '||(q^T KV) / (q^T k_sum + eps)||'),
]

SHOW_STD_SHADING = False
model_order = POSITION_METRIC_EXPERIMENT['model_names']
for metric_col, metric_label in metric_specs:
    plot_position_metric_per_layer(
        position_metric_df,
        metric=metric_col,
        title_prefix=f'{metric_label} vs token position',
        model_order=model_order,
        layer_indices=selected_layers,
        log_x=bool(POSITION_METRIC_EXPERIMENT['log_x']),
        y_scale='log',
        show_std_shading=SHOW_STD_SHADING,
        seqlen=int(POSITION_METRIC_EXPERIMENT['seqlen']),
    )

diagnostic_metric_specs = [
    ('sigma1_share', 'sigma1 / sum(sigma)', False),
    ('update_alignment', 'cos(vec(delta_KV_t), vec(KV_{t-1}))', False),
]

for metric_col, metric_label, use_symlog in diagnostic_metric_specs:
    plot_position_metric_per_layer(
        position_metric_df,
        metric=metric_col,
        title_prefix=f'{metric_label} vs token position',
        model_order=model_order,
        layer_indices=selected_layers,
        log_x=bool(POSITION_METRIC_EXPERIMENT['log_x']),
        y_scale='symlog' if use_symlog else 'linear',
        show_std_shading=SHOW_STD_SHADING,
        seqlen=int(POSITION_METRIC_EXPERIMENT['seqlen']),
    )


## DeltaNet and GLA Gating vs Position

Track the DeltaNet update gate
$$\mathbf{I} - \beta_t \mathbf{k}_t \mathbf{k}_t^\top$$
and the GLA recurrent decay gate
$$\exp(\mathbf{g}_t^k)$$
over token position for representative layers.


In [ ]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from einops import rearrange, repeat
from fla.layers.delta_net import DeltaNet, elu_p1, sum_norm
from fla.layers.gla import GatedLinearAttention

from pfns.experiments.model_benchmarks.benchmark_batch_generators import (
    _set_data_generation_seed,
    create_seq_len_batch_generator,
)
from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import (
    functional_model_config,
    get_autocast_models_from_registry,
)
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.training_utils import (
    categorical_mask_to_inds,
    is_autocast_dtype_enabled,
    move_style_and_check_shape,
    move_y_style_and_check_shape,
)

GATE_POSITION_EXPERIMENT = {
    'name': 'seq_len_gate_position_debug_with_beta',
    'model_names': [
        'DeltaNet_Comb_ST',
        'GLA_Comb_ST',
    ],
    'seqlen': 64_000,
    'num_repetitions': 3,
    'data_generation_seed': EXPERIMENT['data_generation_seed'],
    'num_classes': EXPERIMENT['num_classes'],
    'num_features': EXPERIMENT['num_features'],
    'num_test_samples': EXPERIMENT['num_test_samples'],
    'log_x': True,
}

GATE_POSITION_LAYER_SELECTION = None  # None -> auto-pick first / middle / last
GATE_POSITION_CACHE_OVERWRITE = True


def _run_model_autocast(fn, *, model_name: str, autocast_models: dict[str, torch.dtype], device: str):
    autocast_dtype = autocast_models.get(model_name)
    device_type = torch.device(device).type
    with torch.inference_mode():
        with torch.autocast(
            device_type=device_type,
            enabled=(device_type == 'cuda') and is_autocast_dtype_enabled(autocast_dtype),
            dtype=autocast_dtype or torch.float32,
        ):
            return fn()


def _materialize_seq_len_batch_local(*, experiment: dict, seqlen: int, rep: int, device: str):
    _set_data_generation_seed(int(experiment['data_generation_seed']))
    generator = create_seq_len_batch_generator(
        task_variant='tabular_prior',
        num_batches=rep + 1,
        smallest_seqlen=int(seqlen),
        largest_seqlen=int(seqlen),
        num_features=int(experiment['num_features']),
        num_classes=int(experiment['num_classes']),
        number_of_test_samples=int(experiment['num_test_samples']),
        default_device=device,
        task_kwargs={},
    )
    for current_rep, (batch, _) in enumerate(generator):
        if current_rep == rep:
            return batch
    raise RuntimeError(f'Unable to materialize repetition {rep} for seqlen={seqlen}.')


def _prepare_embedded_train_input_local(model, batch, *, seqlen: int, device: str) -> torch.Tensor:
    x = batch.x[:, :seqlen]
    y = batch.y[:, :seqlen]
    style = move_style_and_check_shape(batch.style, x, device)
    y_style = move_y_style_and_check_shape(batch.y_style, y, device)
    categorical_inds = categorical_mask_to_inds(batch.categorical_mask)
    x_bf, y_bf, _ = model._prepare_batch_first_inputs(x.to(device), y.to(device), None)
    embedded, _, _, _ = model._build_embedded_input(
        x_bf,
        y_bf,
        single_eval_pos=int(y_bf.shape[1]),
        style=style,
        y_style=y_style,
        categorical_inds=categorical_inds,
        cache_trainset_representation=True,
    )
    return embedded


def _track_fla_gate_position_metrics(model, embedded: torch.Tensor, *, model_name: str) -> pd.DataFrame:
    backbone = getattr(model, 'transformer_layers', None)
    if backbone is None or not hasattr(backbone, 'fla'):
        raise TypeError(f'{model_name} is not backed by an FLA backbone.')

    hidden, _ = backbone._prepare_fla_input(embedded)
    rows: list[dict[str, object]] = []
    for layer_idx, block in enumerate(backbone.fla.layers):
        hidden_norm = block.attn_norm(hidden)
        attn_layer = block.attn

        if isinstance(attn_layer, DeltaNet):
            q = attn_layer.q_proj(hidden_norm)
            k = attn_layer.k_proj(hidden_norm)
            if attn_layer.qk_activation == 'silu':
                q, k = F.silu(q), F.silu(k)
            elif attn_layer.qk_activation == 'relu':
                q, k = q.relu(), k.relu()
            elif attn_layer.qk_activation == 'elu':
                q, k = elu_p1(q), elu_p1(k)
            elif attn_layer.qk_activation != 'identity':
                raise NotImplementedError(f'Unsupported DeltaNet qk_activation={attn_layer.qk_activation!r}')

            q = rearrange(q, 'b t (h d) -> b t h d', d=attn_layer.head_k_dim)
            k = rearrange(k, 'b t (h d) -> b t h d', d=attn_layer.head_k_dim)
            if attn_layer.qk_norm == 'sum':
                q = sum_norm(q).to(q)
                k = sum_norm(k).to(k)

            if attn_layer.use_beta:
                beta = attn_layer.b_proj(hidden_norm).sigmoid()
            else:
                beta = torch.ones_like(q[..., 0])
            if attn_layer.allow_neg_eigval:
                beta = beta * 2.0

            beta = beta.float()
            k_sq_norm = k.float().square().sum(dim=-1)
            delta_eig = 1.0 - beta * k_sq_norm
            delta_gate_spectral_norm = torch.maximum(delta_eig.abs(), torch.ones_like(delta_eig))

            for token_idx in range(int(delta_gate_spectral_norm.shape[1])):
                vals = delta_gate_spectral_norm[:, token_idx].reshape(-1)
                beta_vals = beta[:, token_idx].reshape(-1)
                eig_vals = delta_eig[:, token_idx].reshape(-1)
                rows.append(
                    {
                        'model': model_name,
                        'layer_idx': int(layer_idx),
                        'head_idx': -1,
                        'position': int(token_idx + 1),
                        'delta_gate_spectral_norm': float(vals.mean().item()),
                        'delta_gate_spectral_norm_std': float(vals.std(unbiased=False).item()),
                        'delta_gate_eigenvalue': float(eig_vals.mean().item()),
                        'delta_gate_eigenvalue_std': float(eig_vals.std(unbiased=False).item()),
                        'beta_mean': float(beta_vals.mean().item()),
                        'beta_mean_std': float(beta_vals.std(unbiased=False).item()),
                        'gla_decay_mean': float('nan'),
                        'gla_decay_mean_std': float('nan'),
                    }
                )
                for head_idx in range(int(delta_gate_spectral_norm.shape[2])):
                    head_vals = delta_gate_spectral_norm[:, token_idx, head_idx].reshape(-1)
                    head_beta_vals = beta[:, token_idx, head_idx].reshape(-1)
                    head_eig_vals = delta_eig[:, token_idx, head_idx].reshape(-1)
                    rows.append(
                        {
                            'model': model_name,
                            'layer_idx': int(layer_idx),
                            'head_idx': int(head_idx),
                            'position': int(token_idx + 1),
                            'delta_gate_spectral_norm': float(head_vals.mean().item()),
                            'delta_gate_spectral_norm_std': float(head_vals.std(unbiased=False).item()),
                            'delta_gate_eigenvalue': float(head_eig_vals.mean().item()),
                            'delta_gate_eigenvalue_std': float(head_eig_vals.std(unbiased=False).item()),
                            'beta_mean': float(head_beta_vals.mean().item()),
                            'beta_mean_std': float(head_beta_vals.std(unbiased=False).item()),
                            'gla_decay_mean': float('nan'),
                            'gla_decay_mean_std': float('nan'),
                        }
                    )

        elif isinstance(attn_layer, GatedLinearAttention):
            gk = attn_layer.gk_proj(hidden_norm)
            if attn_layer.num_kv_groups > 1:
                gk = repeat(gk, 'b t (h d) -> b t (h g) d', g=attn_layer.num_kv_groups, d=attn_layer.head_k_dim)
            else:
                gk = rearrange(gk, 'b t (h d) -> b t h d', d=attn_layer.head_k_dim)
            gk = F.logsigmoid(gk) / attn_layer.gate_logit_normalizer
            if attn_layer.clamp_min is not None:
                gk = torch.clamp_min(gk, attn_layer.clamp_min)
            gla_decay = gk.float().exp()
            gla_decay_mean = gla_decay.mean(dim=-1)

            for token_idx in range(int(gla_decay_mean.shape[1])):
                vals = gla_decay_mean[:, token_idx].reshape(-1)
                rows.append(
                    {
                        'model': model_name,
                        'layer_idx': int(layer_idx),
                        'head_idx': -1,
                        'position': int(token_idx + 1),
                        'delta_gate_spectral_norm': float('nan'),
                        'delta_gate_spectral_norm_std': float('nan'),
                        'delta_gate_eigenvalue': float('nan'),
                        'delta_gate_eigenvalue_std': float('nan'),
                        'beta_mean': float('nan'),
                        'beta_mean_std': float('nan'),
                        'gla_decay_mean': float(vals.mean().item()),
                        'gla_decay_mean_std': float(vals.std(unbiased=False).item()),
                    }
                )
                for head_idx in range(int(gla_decay_mean.shape[2])):
                    head_vals = gla_decay_mean[:, token_idx, head_idx].reshape(-1)
                    rows.append(
                        {
                            'model': model_name,
                            'layer_idx': int(layer_idx),
                            'head_idx': int(head_idx),
                            'position': int(token_idx + 1),
                            'delta_gate_spectral_norm': float('nan'),
                            'delta_gate_spectral_norm_std': float('nan'),
                            'delta_gate_eigenvalue': float('nan'),
                            'delta_gate_eigenvalue_std': float('nan'),
                            'beta_mean': float('nan'),
                            'beta_mean_std': float('nan'),
                            'gla_decay_mean': float(head_vals.mean().item()),
                            'gla_decay_mean_std': float(head_vals.std(unbiased=False).item()),
                        }
                    )

        hidden = block(
            hidden,
            attention_mask=None,
            past_key_values=None,
            use_cache=False,
            output_attentions=False,
        )[0]

    return pd.DataFrame(rows)


def run_gate_position_metric_tracking(*, experiment: dict, device: str) -> pd.DataFrame:
    model_configs = get_models_from_names(experiment['model_names'])
    models, _ = load_models_for_benchmark(model_configs, device=device)
    autocast_models = get_autocast_models_from_registry(model_configs, device=device)

    rows: list[dict[str, object]] = []
    for rep in tqdm(range(int(experiment['num_repetitions'])), desc='Gate position tracking'):
        base_batch = _materialize_seq_len_batch_local(
            experiment=experiment,
            seqlen=int(experiment['seqlen']),
            rep=rep,
            device=device,
        )
        for raw_name, model in models.items():
            model_name = str(raw_name)
            embedded = _run_model_autocast(
                lambda: _prepare_embedded_train_input_local(
                    model,
                    base_batch,
                    seqlen=int(experiment['seqlen']),
                    device=device,
                ),
                model_name=model_name,
                autocast_models=autocast_models,
                device=device,
            )
            metric_df = _run_model_autocast(
                lambda: _track_fla_gate_position_metrics(
                    model,
                    embedded,
                    model_name=model_name,
                ),
                model_name=model_name,
                autocast_models=autocast_models,
                device=device,
            )
            metric_df['rep'] = int(rep)
            metric_df['seqlen'] = int(experiment['seqlen'])
            rows.extend(metric_df.to_dict(orient='records'))
    return pd.DataFrame(rows)


gate_position_model_configs = get_models_from_names(GATE_POSITION_EXPERIMENT['model_names'])
gate_position_cache_key = experiment_payload_hash(
    experiment_payload={
        'experiment': GATE_POSITION_EXPERIMENT,
        'models_to_compare': {
            model_name: functional_model_config(model_config)
            for model_name, model_config in gate_position_model_configs.items()
        },
    }
)
gate_position_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{GATE_POSITION_EXPERIMENT['name']}_{gate_position_cache_key}.pkl"
)

if gate_position_cache_path.exists() and not GATE_POSITION_CACHE_OVERWRITE:
    gate_position_df = pd.read_pickle(gate_position_cache_path)
    print(f'Loaded gate_position_df from cache: {gate_position_cache_path}')
else:
    gate_position_df = run_gate_position_metric_tracking(
        experiment=GATE_POSITION_EXPERIMENT,
        device=device,
    )
    gate_position_df.to_pickle(gate_position_cache_path)
    print(f'Saved gate_position_df to cache: {gate_position_cache_path}')

print('gate_position_df rows:', len(gate_position_df))
gate_position_df.head()

In [ ]:
def _auto_select_gate_layers(layer_values: list[int], max_layers: int = 5) -> list[int]:
    layer_values = sorted(int(v) for v in layer_values)
    if len(layer_values) <= max_layers:
        return layer_values
    candidate_indices = np.linspace(0, len(layer_values) - 1, num=max_layers, dtype=int).tolist()
    return [layer_values[idx] for idx in sorted(set(candidate_indices))]


def _subsample_gate_curve(layer_df: pd.DataFrame) -> pd.DataFrame:
    positions = layer_df['position'].to_numpy(dtype=int)
    keep = np.zeros(len(layer_df), dtype=bool)
    keep |= positions <= 100
    keep |= ((positions > 100) & (positions <= 1_000) & ((positions % 5) == 0))
    keep |= ((positions > 1_000) & (positions <= 10_000) & ((positions % 20) == 0))
    keep |= ((positions > 10_000) & ((positions % 100) == 0))
    keep[0] = True
    keep[-1] = True
    return layer_df.iloc[keep]


available_gate_layers = sorted(gate_position_df['layer_idx'].astype(int).unique().tolist())
selected_gate_layers = (
    GATE_POSITION_LAYER_SELECTION
    if GATE_POSITION_LAYER_SELECTION is not None
    else _auto_select_gate_layers(available_gate_layers)
)
print('Selected gate layers:', selected_gate_layers)

gate_metric_specs = [
    ('delta_gate_spectral_norm', r'$\|I - \beta_t k_t k_t^\top\|_2$'),
    ('delta_gate_eigenvalue', r'$1 - \beta_t\|k_t\|_2^2$'),
    ('beta_mean', r'$\mathrm{mean}(\beta_t)$'),
    ('gla_decay_mean', r'$\mathrm{mean}(\exp(g_t^k))$'),
]

GATE_PLOT_MODE = 'layer_mean'  # 'layer_mean' or 'per_head'
GATE_PER_HEAD_LAYER = 6  # defaults to middle selected layer
GATE_HEAD_SELECTION = None  # optional list of head indices

SHOW_GATE_STD_SHADING = False
SHOW_GATE_REP_LINES = True
GATE_REP_ALPHA = 0.18
SHOW_GATE_REP_MEAN = True

gate_model_order = GATE_POSITION_EXPERIMENT['model_names']
per_head_layer = (
    int(GATE_PER_HEAD_LAYER)
    if GATE_PER_HEAD_LAYER is not None
    else int(selected_gate_layers[len(selected_gate_layers) // 2])
)
curve_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for metric_col, metric_label in gate_metric_specs:
    visible_models = []
    for model_name in gate_model_order:
        sub = gate_position_df[
            (gate_position_df['model'] == model_name)
            & gate_position_df[metric_col].notna()
        ].copy()
        if GATE_PLOT_MODE == 'layer_mean':
            sub = sub[
                (sub['head_idx'] == -1)
                & sub['layer_idx'].isin(selected_gate_layers)
            ]
        else:
            sub = sub[
                (sub['head_idx'] != -1)
                & (sub['layer_idx'] == per_head_layer)
            ]
            if GATE_HEAD_SELECTION is not None:
                sub = sub[sub['head_idx'].isin({int(v) for v in GATE_HEAD_SELECTION})]
        if not sub.empty:
            visible_models.append(model_name)
    if not visible_models:
        continue
    fig, axes = plt.subplots(1, len(visible_models), figsize=(7 * len(visible_models), 4), sharey=False)
    if len(visible_models) == 1:
        axes = [axes]
    for ax, model_name in zip(axes, visible_models):
        sub = gate_position_df[
            (gate_position_df['model'] == model_name)
            & gate_position_df[metric_col].notna()
        ].copy()
        if GATE_PLOT_MODE == 'layer_mean':
            sub = sub[
                (sub['head_idx'] == -1)
                & sub['layer_idx'].isin(selected_gate_layers)
            ]
            group_cols = ['layer_idx', 'position']
            sort_cols = ['layer_idx', 'position']
            rep_group_cols = ['layer_idx', 'rep', 'position']
            rep_sort_cols = ['layer_idx', 'rep', 'position']
            curve_key = 'layer_idx'
            curve_values = selected_gate_layers
        else:
            sub = sub[
                (sub['head_idx'] != -1)
                & (sub['layer_idx'] == per_head_layer)
            ]
            if GATE_HEAD_SELECTION is not None:
                sub = sub[sub['head_idx'].isin({int(v) for v in GATE_HEAD_SELECTION})]
            group_cols = ['head_idx', 'position']
            sort_cols = ['head_idx', 'position']
            rep_group_cols = ['head_idx', 'rep', 'position']
            rep_sort_cols = ['head_idx', 'rep', 'position']
            curve_key = 'head_idx'
            curve_values = sorted(sub['head_idx'].astype(int).unique().tolist())
        curve_color_map = {
            int(curve_idx): curve_colors[idx % len(curve_colors)]
            for idx, curve_idx in enumerate(curve_values)
        }
        if SHOW_GATE_REP_LINES:
            rep_summary = (
                sub.groupby(rep_group_cols, observed=True)
                .agg(mean=(metric_col, 'mean'))
                .reset_index()
                .sort_values(rep_sort_cols)
            )
            seen_labels = set()
            for (curve_idx, rep_idx), rep_df in rep_summary.groupby([curve_key, 'rep'], observed=True):
                rep_df = _subsample_gate_curve(rep_df)
                x = rep_df['position'].to_numpy()
                y = rep_df['mean'].to_numpy()
                label = f'L{int(curve_idx)}' if GATE_PLOT_MODE == 'layer_mean' else f'H{int(curve_idx)}'
                line_label = label if not SHOW_GATE_REP_MEAN and int(curve_idx) not in seen_labels else '_nolegend_'
                seen_labels.add(int(curve_idx))
                ax.plot(
                    x,
                    y,
                    label=line_label,
                    linewidth=1.0,
                    alpha=GATE_REP_ALPHA,
                    color=curve_color_map.get(int(curve_idx)),
                )
        if SHOW_GATE_REP_MEAN:
            summary = (
                sub.groupby(group_cols, observed=True)
                .agg(
                    mean=(metric_col, 'mean'),
                    std=(f'{metric_col}_std', 'mean'),
                )
                .reset_index()
                .sort_values(sort_cols)
            )
            for curve_idx, layer_df in summary.groupby(curve_key, observed=True):
                layer_df = _subsample_gate_curve(layer_df)
                x = layer_df['position'].to_numpy()
                y = layer_df['mean'].to_numpy()
                y_std = layer_df['std'].to_numpy()
                label = f'L{int(curve_idx)}' if GATE_PLOT_MODE == 'layer_mean' else f'H{int(curve_idx)}'
                color = curve_color_map.get(int(curve_idx))
                ax.plot(x, y, label=label, linewidth=1.8, color=color)
                if SHOW_GATE_STD_SHADING:
                    ax.fill_between(x, y - y_std, y + y_std, alpha=0.15, color=color)
        if GATE_POSITION_EXPERIMENT['log_x']:
            ax.set_xscale('log')
        title = str(model_name) if GATE_PLOT_MODE == 'layer_mean' else f'{model_name} | L{per_head_layer}'
        ax.set_title(title)
        ax.set_xlabel('Token position')
        ax.set_ylabel(metric_label)
        ax.grid(True, alpha=0.3, which='both')
        if SHOW_GATE_REP_LINES or SHOW_GATE_REP_MEAN:
            ax.legend(loc='best', fontsize=8)
    fig.suptitle(f'{metric_label} vs position (seqlen={GATE_POSITION_EXPERIMENT["seqlen"]})', x=0.5, ha='center')
    fig.tight_layout(rect=(0, 0, 1, 0.96))
